In [ ]:
# ===== NB16 | CELL 1: header + GPU check =====
from google.colab import drive
drive.mount('/content/drive')
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change the runtime to T4 GPU')
print('CELL 1 DONE')

In [ ]:
# ===== NB16 | CELL 2: counterfactual fairness test =====
# WHAT: 20 harmless sentence templates x 25 identity terms + 5 neutral control words = 600 sentences, for example
#       'My friend is gay.' None is toxic, so the ideal flag rate is 0. We score them with the frozen system,
#       the best single DistilBERT and the Logistic Regression baseline.
# WHY:  the brief asks whether the model flags a comment just for mentioning an identity group. If identity
#       words are flagged far more often than neutral words, the model keys on the word, not on toxicity.
import sys, importlib, json, os
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib, evallib, runlib
for m in (toxlib, evallib, runlib): importlib.reload(m)
from toxlib import LABELS, load_split, load_probs, tune_thresholds
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
for d in ['graphs', 'tables', 'fairness']:
    os.makedirs(f'{PROJECT_DIR}/results/{d}', exist_ok=True)

AXES = {'religion': ['muslim', 'hindu', 'christian', 'jewish', 'sikh', 'buddhist', 'atheist'],
        'sexual orientation / gender identity': ['gay', 'lesbian', 'bisexual', 'homosexual', 'transgender', 'straight'],
        'race / nationality': ['black', 'white', 'asian', 'latino', 'indian', 'american'],
        'caste': ['dalit', 'brahmin'], 'disability': ['blind', 'deaf', 'disabled'], 'age': ['old'],
        'control (neutral words)': ['tall', 'tired', 'happy', 'busy', 'local']}
CAP = {'muslim': 'Muslim', 'hindu': 'Hindu', 'christian': 'Christian', 'jewish': 'Jewish', 'sikh': 'Sikh', 'buddhist': 'Buddhist',
       'indian': 'Indian', 'american': 'American', 'asian': 'Asian', 'latino': 'Latino', 'black': 'Black', 'white': 'White',
       'dalit': 'Dalit', 'brahmin': 'Brahmin'}
TEMPLATES = ['I am a {} person.', 'My friend is {}.', 'I love my {} neighbors.', '{} people are welcome here.',
             'She works with a {} colleague.', 'He is a {} teacher at our school.', 'We had dinner with a {} family.',
             'The {} student won the prize.', 'My neighbor is {} and very kind.', 'Many {} people live in this city.',
             'I respect my {} classmates.', 'They hired a {} engineer.', 'A {} doctor treated my mother.',
             'Our team includes a {} member.', 'This book was written by a {} author.', 'I met a {} man at the market.',
             'She is proud to be {}.', 'He grew up in a {} household.', 'The {} community organised a festival.',
             'I talked to a {} woman on the bus.']
rows = []
for axis, terms in AXES.items():
    for t in terms:
        for k, tp in enumerate(TEMPLATES):
            s = tp.format(CAP.get(t, t)); rows.append({'term': t, 'axis': axis, 'template': k, 'text': s[0].upper() + s[1:]})
S = pd.DataFrame(rows)
print(f'{len(S)} harmless test sentences ({S.term.nunique()} terms x {len(TEMPLATES)} templates). Example:', S.text.iloc[0])

cfg = json.load(open(f'{PROJECT_DIR}/results/final/final_config.json'))
dec = json.load(open(f'{PROJECT_DIR}/results/experiment_logs/loss_decision.json'))
Yva = load_split(PROJECT_DIR, 'val')[LABELS].values
systems = {'final_ensemble': (cfg['runs'], cfg['thresholds'])}
for name, run in [('best_distilbert', dec['winner_run']), ('logreg_A1', 'A1')]:
    systems[name] = ([run], tune_thresholds(Yva, load_probs(PROJECT_DIR, run, 'val')))

rng = np.random.RandomState(42)
def ci_mean(x, n=1000):
    x = np.asarray(x, dtype=float); b = [x[rng.randint(0, len(x), len(x))].mean() for _ in range(n)]
    return np.percentile(b, [2.5, 97.5])

all_rows, scores = [], S.copy()
for name, (runs, thr) in systems.items():
    ens = runlib.Ensemble(PROJECT_DIR, runs)
    Pm = ens.predict(S.text.tolist()); ens.free(); del ens
    scores[f'{name}_p_toxic'] = Pm[:, 0]; scores[f'{name}_p_identity_hate'] = Pm[:, 5]
    S['p_toxic'], S['p_ih'] = Pm[:, 0], Pm[:, 5]
    S['f_toxic'] = Pm[:, 0] >= thr['toxic']; S['f_ih'] = Pm[:, 5] >= thr['identity_hate']
    ctrl = S[S.axis.str.startswith('control')]
    for term, g in S.groupby('term', sort=False):
        g = g.sort_values('template')
        lo_t, hi_t = ci_mean(g.f_toxic.values); lo_i, hi_i = ci_mean(g.f_ih.values)
        all_rows.append({'system': name, 'term': term, 'axis': g.axis.iloc[0], 'mean_p_toxic': round(g.p_toxic.mean(), 3),
                         'toxic_flag_rate': round(g.f_toxic.mean(), 3), 'toxic_flag_CI': f'{lo_t:.2f}-{hi_t:.2f}',
                         'mean_p_identity_hate': round(g.p_ih.mean(), 3), 'identity_hate_flag_rate': round(g.f_ih.mean(), 3),
                         'identity_hate_flag_CI': f'{lo_i:.2f}-{hi_i:.2f}',
                         'control_toxic_flag_rate': round(ctrl.f_toxic.mean(), 3), 'control_identity_hate_flag_rate': round(ctrl.f_ih.mean(), 3)})
R = pd.DataFrame(all_rows)
R.to_csv(f'{PROJECT_DIR}/results/tables/T11_counterfactual_by_term.csv', index=False)
scores.to_csv(f'{PROJECT_DIR}/results/fairness/counterfactual_scores.csv', index=False)

F = R[R.system == 'final_ensemble'].sort_values('identity_hate_flag_rate', ascending=False)
print('\nT11 final system, harmless sentences (ideal flag rate = 0). Sorted by identity_hate flag rate:')
print(F[['term', 'axis', 'mean_p_toxic', 'toxic_flag_rate', 'toxic_flag_CI', 'mean_p_identity_hate', 'identity_hate_flag_rate', 'identity_hate_flag_CI']].to_string(index=False))
print('Control words (neutral): toxic flag rate', F.control_toxic_flag_rate.iloc[0], '| identity_hate flag rate', F.control_identity_hate_flag_rate.iloc[0])

T11b = R.groupby(['system', 'axis'])[['toxic_flag_rate', 'identity_hate_flag_rate']].mean().round(3).reset_index()
T11b.to_csv(f'{PROJECT_DIR}/results/tables/T11b_counterfactual_by_axis.csv', index=False)
print('\nT11b mean identity_hate flag rate by axis and system:'); print(T11b.pivot(index='axis', columns='system', values='identity_hate_flag_rate').round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
cols = dict(zip(AXES, plt.cm.tab10.colors))
for a, (col, ci_col, title) in zip(axes, [('toxic_flag_rate', 'toxic_flag_CI', 'flagged as toxic'), ('identity_hate_flag_rate', 'identity_hate_flag_CI', 'flagged as identity_hate')]):
    d = F.sort_values(col); lo = np.array([float(c.split('-')[0]) for c in d[ci_col]]); hi = np.array([float(c.split('-')[1]) for c in d[ci_col]])
    a.barh(d.term, d[col], xerr=[d[col] - lo, hi - d[col]], color=[cols[x] for x in d.axis], capsize=2)
    a.axvline(d['control_' + col].iloc[0], color='k', ls='--', lw=1); a.set_title(f'% of harmless sentences {title}'); a.set_xlim(0, 1)
plt.suptitle('F17: counterfactual test (final system), dashed line = neutral control words'); plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/results/graphs/F17_counterfactual.png', dpi=150); plt.show()
print('CELL 2 DONE')

In [ ]:
# ===== NB16 | CELL 3: fairness on real test comments =====
# WHAT: on the pooled internal + external test comments (about 88k), find comments containing each identity term,
#       and measure (1) the false-flag rate on NON-toxic ones relative to all non-toxic comments,
#       (2) subgroup AUC, BPSN AUC and BNSP AUC, all with 95% bootstrap intervals.
# WHY:  the templates test controlled sentences, this tests real language. Both are needed to support a claim.
# LIMIT: terms are found by keyword (the dataset has no identity labels), so words like 'white' or 'black'
#        also appear in non-identity uses. Say this in the report.
import sys, importlib, json, os
PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
sys.path.insert(0, f'{PROJECT_DIR}/src')
import toxlib, evallib
for m in (toxlib, evallib): importlib.reload(m)
from toxlib import LABELS, load_split
from sklearn.metrics import roc_auc_score
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
for d in ['graphs', 'tables', 'fairness']:
    os.makedirs(f'{PROJECT_DIR}/results/{d}', exist_ok=True)

cfg = json.load(open(f'{PROJECT_DIR}/results/final/final_config.json')); thr_t = cfg['thresholds']['toxic']
parts = []
for split in ['test_internal', 'test_external']:
    d = load_split(PROJECT_DIR, split); d['p'] = np.load(f'{PROJECT_DIR}/results/final/final_{split}.npy')[:, 0]; parts.append(d)
D = pd.concat(parts, ignore_index=True)
y = D.toxic.values.astype(int); s = D.p.values; flag = s >= thr_t; nontox = y == 0
overall_fpr = flag[nontox].mean(); overall_auc = roc_auc_score(y, s)
print(f'Pooled test comments: {len(D)} | non-toxic: {int(nontox.sum())} | overall false-flag rate: {overall_fpr:.3f} | overall toxic AUC: {overall_auc:.3f}')

TERMS = ['muslim', 'hindu', 'christian', 'jewish', 'sikh', 'buddhist', 'atheist', 'gay', 'lesbian', 'bisexual', 'homosexual',
         'transgender', 'straight', 'black', 'white', 'asian', 'latino', 'indian', 'american', 'dalit', 'brahmin', 'blind', 'deaf',
         'disabled', 'old']
rng = np.random.RandomState(42)
rows = []
for t in TERMS:
    mask = D.comment_text.fillna('').str.contains(rf'\b{t}\b', case=False, regex=True).values
    n = int(mask.sum()); sel = mask & nontox; nn = int(sel.sum())
    if n < 30 or nn < 20:
        rows.append({'term': t, 'n_comments': n, 'n_nontoxic': nn, 'note': 'too few comments'}); continue
    fpr = flag[sel].mean(); f = flag[sel].astype(float)
    boots = [f[rng.randint(0, nn, nn)].mean() / overall_fpr for _ in range(500)]
    sub, bpsn, bnsp = evallib.subgroup_aucs(y, s, mask); lo, hi = evallib.bpsn_ci(y, s, mask)
    rows.append({'term': t, 'n_comments': n, 'n_nontoxic': nn, 'false_flag_rate': round(fpr, 3), 'x_overall': round(fpr / overall_fpr, 2),
                 'x_overall_95CI': f'{np.percentile(boots, 2.5):.1f}-{np.percentile(boots, 97.5):.1f}',
                 'subgroup_AUC': round(sub, 3), 'BPSN_AUC': round(bpsn, 3), 'BPSN_95CI': f'{lo:.3f}-{hi:.3f}', 'BNSP_AUC': round(bnsp, 3)})
T12 = pd.DataFrame(rows).sort_values('x_overall', ascending=False)
T12.to_csv(f'{PROJECT_DIR}/results/tables/T12_real_comment_fairness.csv', index=False)
print('\nT12 real comments: false-flag rate on NON-toxic comments containing the word, relative to all non-toxic comments')
print('(x_overall 1.0 = treated like any other comment; BPSN AUC well below the overall AUC means the word itself pushes scores up)')
print(T12.to_string(index=False))

V = T12.dropna(subset=['BPSN_AUC']).sort_values('BPSN_AUC')
lo = np.array([float(c.split('-')[0]) for c in V.BPSN_95CI]); hi = np.array([float(c.split('-')[1]) for c in V.BPSN_95CI])
plt.figure(figsize=(7, 7)); plt.barh(V.term, V.BPSN_AUC, xerr=[V.BPSN_AUC - lo, hi - V.BPSN_AUC], capsize=2, color='#4C72B0')
plt.axvline(overall_auc, color='r', ls='--', label=f'overall AUC {overall_auc:.3f}'); plt.xlim(0.5, 1.0)
plt.xlabel('BPSN AUC (lower = the word itself raises the toxic score)'); plt.legend(); plt.title('F18: BPSN AUC by identity term (real test comments)')
plt.tight_layout(); plt.savefig(f'{PROJECT_DIR}/results/graphs/F18_bpsn_auc.png', dpi=150); plt.show()
print('CELL 3 DONE')